In [ ]:
import sys, os
import asyncio
import json, re
import time
import itertools
import uuid
import pandas as pd
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple, Callable
import numpy as np

In [ ]:
@dataclass
class ModelConfig:
    client: Any                   # OpenAI / Claude / vLLM client
    api_method: Callable          # chat.completions.create-like method
    use_system_message: bool
    args: Optional[Dict[str, Any]] = None

async def call_llm(model: ModelConfig, messages: List[Dict[str, str]]):
    """
    Returns: (text, token_usage)
    """
    call_args = {
        "messages": messages,
    }
    if model.args is not None:
        call_args.update(model.args)
    response = await model.api_method(**call_args)

    text = response.choices[0].message.content
    usage = getattr(response, "usage", {}) or {}

    return text, {
        "prompt_tokens": usage.prompt_tokens,
        "completion_tokens": usage.completion_tokens,
        "total_tokens": usage.total_tokens,
    }

In [ ]:
@dataclass
class BargainingTask:
    item_name: str
    item_description: str

    buyer_persona: str
    seller_persona: str

    buyer_res_price: float
    seller_res_price: float

    buyer_res_price_range: Optional[Tuple[float, float]] = None
    seller_res_price_range: Optional[Tuple[float, float]] = None

    transparency: str = "full"   # full | buyer_unaware | seller_unaware | both_unaware
    mode: str = "sequential"     # sequential | simultaneous
    max_rounds: int = 5
    first_actor: str = "buyer"        # buyer | seller, used only in sequential mode

In [ ]:
def build_system_prompt(role: str, task: BargainingTask) -> str:
    prompt = f"""
You are the {role.upper()} in a bargaining negotiation.

Item: {task.item_name}
Description: {task.item_description}

Your persona: {task.buyer_persona if role=='buyer' else task.seller_persona}
Your reservation price: {task.buyer_res_price if role=='buyer' else task.seller_res_price}. You can always {'buy from' if (role=='buyer') else 'sell to'} the market at this price if the bargaining fails.
"""

    # Transparency
    if task.transparency == "full":
        opp = task.seller_res_price if role == "buyer" else task.buyer_res_price
        prompt += f"You know the other agent's reservation price is {opp}.\n"
    elif task.transparency == "buyer_unaware":
        lo, hi = task.seller_res_price_range
        if role == "seller":
            prompt += f"You know the buyer's reservation price is {task.buyer_res_price}.\n"
            prompt += f"The buyer does not know your exact reservation price, their prior on your reservation price is ~ Uniform[{lo}, {hi}].\n"
        else:
            prompt += f"Your prior on the seller's reservation price is ~ Uniform[{lo}, {hi}].\n"
    elif task.transparency == "seller_unaware":
        lo, hi = task.buyer_res_price_range
        if role == "buyer":
            prompt += f"You know the seller's reservation price is {task.seller_res_price}.\n"
            prompt += f"The seller does not know your exact reservation price, their prior on your reservation price is ~ Uniform[{lo}, {hi}].\n"
        else:
            lo, hi = task.buyer_res_price_range
            prompt += f"Your prior on the buyer's reservation price is ~ Uniform[{lo}, {hi}].\n"
    else:
        lo_b, hi_b = task.buyer_res_price_range
        lo_s, hi_s = task.seller_res_price_range
        if role == "seller":
            prompt += f"Your prior on the buyer's reservation price is ~ Uniform[{lo_b}, {hi_b}].\n"
            prompt += f"The buyer does not know your exact reservation price, their prior on your reservation price is ~ Uniform[{lo_s}, {hi_s}].\n"
        else:
            prompt += f"Your prior on the seller's reservation price is ~ Uniform[{lo_s}, {hi_s}].\n"
            prompt += f"The seller does not know your exact reservation price, their prior on your reservation price is ~ Uniform[{lo_b}, {hi_b}].\n"

    prompt += f"""
Output format:

- Write 1–3 sentences of describing your barganing strategy. This will not be exposed to the other agent
- Then output a JSON dict inside a code block using triple backticks:

```json
{{
  "message": "your message to the other agent",
  "action": "OFFER{' / DEAL' if (task.mode == 'sequential') else ''} / NO_DEAL",
  "offer_price": 123.45   # only required if action is OFFER
}}
```
"""

    return prompt.strip()

In [ ]:
import warnings

def parse_output_json_block(text: str):
    """
    Parses the LLM output: free-form thought + JSON code block.
    Validates consistency between action and offer_price.
    
    Returns:
        thought (str)
        message (str)
        action (str) -> "OFFER", "DEAL", "NO_DEAL", "INVALID"
        offer_price (float or None)
    """
    thought = ""
    message = ""
    action = "INVALID"
    offer_price = None

    # Extract JSON code block
    json_block = re.search(r"```json\s*(\{.*?\})\s*```", text, re.DOTALL)
    if not json_block:
        warnings.warn("No JSON block found in LLM output. Marking as INVALID.")
        thought = text.strip()
        return thought, message, action, offer_price

    # Extract JSON
    json_text = json_block.group(1)
    try:
        data = json.loads(json_text)
    except json.JSONDecodeError:
        warnings.warn("JSON block could not be parsed. Marking as INVALID.")
        thought = text.split("```json")[0].strip()
        return thought, message, action, offer_price

    # Extract fields
    thought = text.split("```json")[0].strip()
    message = data.get("message", "")
    action = data.get("action", "INVALID")
    offer_price = data.get("offer_price")

    # Validation
    valid_actions = {"OFFER", "DEAL", "NO_DEAL"}
    if action not in valid_actions:
        warnings.warn(f"Invalid action '{action}' from LLM. Marking as INVALID.")
        action = "INVALID"
        offer_price = None

    # DEAL only allowed in sequential mode (enforced elsewhere in simulator)
    if action == "DEAL":
        offer_price = None  # DEAL doesn't use price

    if action != "OFFER" and offer_price is not None:
        # LLM provided offer_price when not offering → ignore it
        warnings.warn(f"LLM provided offer_price {offer_price} but action is '{action}'. Ignoring offer_price.")
        offer_price = None

    if action == "OFFER":
        # Ensure offer_price is a valid number
        if offer_price is None or not isinstance(offer_price, (int, float)):
            warnings.warn(f"Action is 'OFFER' but offer_price is invalid: {offer_price}. Marking as INVALID.")
            action = "INVALID"
            offer_price = None

    return thought, message, action, offer_price


In [ ]:
def compute_game_theory_benchmarks(task: BargainingTask):
    rB = task.buyer_res_price
    rS = task.seller_res_price

    # True Nash Bargaining Solution
    true_nbs = (rB + rS) / 2

    # Expected Nash Bargaining Solution
    if task.transparency == "full":
        expected_nbs = true_nbs
    else:
        BL, BH = task.buyer_res_price_range or (rB, rB)
        SL, SH = task.seller_res_price_range or (rS, rS)
        expected_nbs = (BL + BH + SL + SH) / 4.0

    # Chatterjee–Samuelson one-shot solution
    cs_solution = -1 # disabled

    return {
        "true_nbs_price": true_nbs,
        "expected_nbs_price": expected_nbs,
        "chatterjee_samuelson_price": cs_solution
    }

In [ ]:
class BargainingSimulator:
    def __init__(self, task, buyer_model, seller_model,
                 jsonl_path="bargaining.jsonl",
                 use_system_message=True):
        self.task = task
        self.buyer_model = buyer_model
        self.seller_model = seller_model
        self.jsonl_path = jsonl_path

        self.buyer_msgs = []
        self.seller_msgs = []
        self.log = []
        with open(self.jsonl_path, "w") as f:
            pass

        self.last_buyer_msg = None
        self.last_seller_msg = None
        self.last_buyer_offer = None
        self.last_seller_offer = None

        self._init_messages()

    def _init_messages(self):
        self.buyer_sys_msg = build_system_prompt("buyer", self.task)
        self.seller_sys_msg = build_system_prompt("seller", self.task)

    def log_event(self, event: dict):
        self.log.append(event)
        with open(self.jsonl_path, "a") as f:
            f.write(json.dumps(event)+"\n")

    async def run(self):
        self.buyer_msgs = [{"role": "system", "content": self.buyer_sys_msg}] if self.buyer_model.use_system_message else []
        self.seller_msgs = [{"role": "system", "content": self.seller_sys_msg}] if self.seller_model.use_system_message else []
        
        for r in range(1, self.task.max_rounds + 1):
            rounds_left = self.task.max_rounds - r

            if self.task.mode == "sequential":
                actor = self.task.first_actor if r % 2 == 1 else ("seller" if self.task.first_actor == "buyer" else "buyer")
                msgs = self.buyer_msgs if actor=="buyer" else self.seller_msgs
                model = self.buyer_model if actor=="buyer" else self.seller_model

                # Compose user message
                content_parts = []
                opp_actor = "Seller" if actor=="buyer" else "Buyer"
                last_opp_msg = self.last_seller_msg if actor=="buyer" else self.last_buyer_msg
                last_opp_offer = self.last_seller_offer if actor=="buyer" else self.last_buyer_offer
                use_sys_msg = self.buyer_model.use_system_message if actor=="buyer" else self.seller_model.use_system_message
                sys_msg = self.buyer_sys_msg if actor=="buyer" else self.seller_sys_msg
                if not use_sys_msg:
                   content_parts.append(sys_msg+"\n") 
                if last_opp_msg:
                    content_parts.append(f"{opp_actor} said: {last_opp_msg}")
                if last_opp_offer:
                    content_parts.append(f"{opp_actor}'s offer: {last_opp_offer}")
                content_parts.append(f"Round {r}, rounds left {rounds_left}")
                
                msgs.append({"role":"user","content":"\n".join(content_parts)})

                # Call LLM
                raw, usage = await call_llm(model, msgs)
                msgs.append({"role":"assistant","content":raw})

                thought, message, action, offer_price = parse_output_json_block(raw)

                # Update last messages
                if actor=="buyer":
                    self.last_buyer_msg = message
                    self.last_buyer_offer = offer_price if action=="OFFER" else self.last_buyer_offer
                else:
                    self.last_seller_msg = message
                    self.last_seller_offer = offer_price if action=="OFFER" else self.last_seller_offer

                # Log
                self.log_event({
                    "round": r,
                    "actor": actor,
                    "thought": thought,
                    "message": message,
                    "action": action,
                    "offer_price": offer_price,
                    "tokens": usage,
                    "raw_msgs": msgs
                })

                # Handle deal / no_deal
                if action == "DEAL":
                    deal_price = self.last_seller_offer if actor=="buyer" else self.last_buyer_offer
                    return self.finalize(deal_price, r)
                if action == "NO_DEAL":
                    break

            else:  # simultaneous
                for actor in ["buyer","seller"]:
                    msgs = self.buyer_msgs if actor=="buyer" else self.seller_msgs
                    model = self.buyer_model if actor=="buyer" else self.seller_model

                    content_parts = []
                    opp_actor = "Seller" if actor=="buyer" else "Buyer"
                    last_opp_msg = self.last_seller_msg if actor=="buyer" else self.last_buyer_msg
                    last_opp_offer = self.last_seller_offer if actor=="buyer" else self.last_buyer_offer
                    use_sys_msg = self.buyer_model.use_system_message if actor=="buyer" else self.seller_model.use_system_message
                    sys_msg = self.buyer_sys_msg if actor=="buyer" else self.seller_sys_msg
                    if not use_sys_msg:
                        content_parts.append(sys_msg+"\n") 
                    if last_opp_msg:
                        content_parts.append(f"{opp_actor} said: {last_opp_msg}")
                    if last_opp_offer:
                        content_parts.append(f"{opp_actor}'s offer: {last_opp_offer}")
                    content_parts.append(f"Round {r}, rounds left {rounds_left}")


                    msgs.append({"role":"user","content":"\n".join(content_parts)})

                # Call both LLMs concurrently
                (b_raw,b_usage),(s_raw,s_usage) = await asyncio.gather(
                    call_llm(self.buyer_model,self.buyer_msgs),
                    call_llm(self.seller_model,self.seller_msgs)
                )

                self.buyer_msgs.append({"role":"assistant","content":b_raw})
                self.seller_msgs.append({"role":"assistant","content":s_raw})

                b_thought,b_msg,b_action,b_price = parse_output_json_block(b_raw)
                s_thought,s_msg,s_action,s_price = parse_output_json_block(s_raw)

                self.last_buyer_msg = b_msg
                self.last_seller_msg = s_msg
                self.last_buyer_offer = b_price if b_action=="OFFER" else self.last_buyer_offer
                self.last_seller_offer = s_price if s_action=="OFFER" else self.last_seller_offer

                self.log_event({"round":r,"actor":"buyer","thought":b_thought,"message":b_msg,"action":b_action,"offer_price":b_price,"tokens":b_usage, "raw_msgs":self.buyer_msgs})
                self.log_event({"round":r,"actor":"seller","thought":s_thought,"message":s_msg,"action":s_action,"offer_price":s_price,"tokens":s_usage, "raw_msgs":self.seller_msgs})

                if b_action=="OFFER" and s_action=="OFFER" and b_price >= s_price:
                    return self.finalize((b_price+s_price)/2,r)
                if b_action=="NO_DEAL" or s_action=="NO_DEAL":
                    break

        # No deal after max rounds
        return self.finalize(None, self.task.max_rounds)

    def finalize(self, deal_price, rounds):
        if deal_price is None:
            buyer_u = seller_u = 0
            result = "no_deal"
        else:
            buyer_u = self.task.buyer_res_price - deal_price
            seller_u = deal_price - self.task.seller_res_price
            result = "deal"

        benchmarks = compute_game_theory_benchmarks(self.task)
        summary = {
            "result": result,
            "deal_price": deal_price,
            "rounds": rounds,
            "buyer_utility": buyer_u,
            "seller_utility": seller_u,
            **benchmarks
        }
        self.log_event({"type":"summary", **summary})
        return summary


In [ ]:
def run_async(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        loop = None

    if loop and loop.is_running():
        return coro  # Jupyter: return awaitable
    else:
        return asyncio.run(coro)

In [ ]:
# OpenAI test

# OpenAI key is read from the OPENAI_API_KEY environment variable (see README).
# OpenAI key is read from the OPENAI_API_KEY environment variable (see README).

if os.environ.get("OPENAI_API_KEY") is None:
        raise RuntimeError("Set the OPENAI_API_KEY environment variable (see README)")

import openai
client = openai.AsyncOpenAI()

buyer_model = ModelConfig(
    client=client,
    api_method=client.chat.completions.create,
    use_system_message=False,
    args={
        "model": "gpt-5.2",
        "temperature": 1.0,
        "max_completion_tokens": 2048
    }
)

seller_model = ModelConfig(
    client=client,
    api_method=client.chat.completions.create,
    use_system_message=False,
    args={
        "model": "gpt-5.2",
        "temperature": 1.0,
        "max_completion_tokens": 2048
    }
)

task = BargainingTask(
    item_name="Vintage Camera",
    item_description="1960s 35mm film camera",
    buyer_persona="Careful hobbyist",
    seller_persona="Experienced collector",
    buyer_res_price=120,
    seller_res_price=80,
    buyer_res_price_range=(100, 140),
    seller_res_price_range=(60, 100),
    transparency="buyer_unaware",
    mode="simultaneous",
    max_rounds=6,
    first_actor="seller"
)

sim = BargainingSimulator(task, buyer_model, seller_model, jsonl_path="./log_0.jsonl", use_system_message=False)
result = await run_async(sim.run())
result


In [ ]:
# ===========================
# Multi-trial runner (correct)
# ===========================

import uuid
import pandas as pd

async def run_trials(
    base_task: BargainingTask,
    buyer_model: ModelConfig,
    seller_model: ModelConfig,
    *,
    modes=("sequential", "simultaneous"),
    transparencies=("full", "buyer_unaware", "seller_unaware", "both_unaware"),
    max_rounds_list=(3, 5, 7),
    first_actors=("buyer", "seller"),
    n_trials=3,
    log_dir="logs"
):
    rows = []

    for mode in modes:
        for transparency in transparencies:
            for max_rounds in max_rounds_list:
                for trial in range(n_trials):

                    # sequential: vary first_actor
                    actor_list = first_actors if mode == "sequential" else [base_task.first_actor]

                    for first_actor in actor_list:
                        task = BargainingTask(
                            item_name=base_task.item_name,
                            item_description=base_task.item_description,
                            buyer_persona=base_task.buyer_persona,
                            seller_persona=base_task.seller_persona,
                            buyer_res_price=base_task.buyer_res_price,
                            seller_res_price=base_task.seller_res_price,
                            buyer_res_price_range=base_task.buyer_res_price_range,
                            seller_res_price_range=base_task.seller_res_price_range,
                            transparency=transparency,
                            mode=mode,
                            max_rounds=max_rounds,
                            first_actor=first_actor
                        )

                        trial_id = str(uuid.uuid4())[:8]
                        jsonl_path = f"{log_dir}/trial_{trial_id}.jsonl"

                        sim = BargainingSimulator(
                            task=task,
                            buyer_model=buyer_model,
                            seller_model=seller_model,
                            jsonl_path=jsonl_path
                        )

                        summary = await sim.run()

                        rows.append({
                            "trial_id": trial_id,
                            "mode": mode,
                            "transparency": transparency,
                            "max_rounds": max_rounds,
                            "first_actor": first_actor,
                            "deal": summary["result"],
                            "deal_price": summary["deal_price"],
                            "buyer_utility": summary["buyer_utility"],
                            "seller_utility": summary["seller_utility"],
                            "true_nash_price": summary["true_nbs_price"],
                            "expected_nash_price": summary["expected_nbs_price"],
                            "cs_price": summary["chatterjee_samuelson_price"],
                            "jsonl_path": jsonl_path
                        })

    return pd.DataFrame(rows)


In [ ]:
!mkdir logs/run_0

In [ ]:
df = await run_trials(
    base_task=task,
    buyer_model=buyer_model,
    seller_model=seller_model,
    modes=("simultaneous", "sequential"),
    transparencies=("buyer_unaware", "both_unaware"),
    max_rounds_list=(3, 6),
    first_actors=("buyer", "seller"),
    n_trials=5,
    log_dir="./logs/run_0"
)

df

In [ ]:
raw_df = df.copy(deep=True)
raw_df.to_json(
    "./results/run_0_raw_trials.jsonl",
    orient="records",
    lines=True
)

In [ ]:
# Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

In [ ]:
# Convert deal outcome to numeric indicator
df = df.copy()

def deal_to_int(x):
    if isinstance(x, bool):
        return int(x)
    if isinstance(x, str):
        return 1 if x.lower() in {"deal", "true", "yes"} else 0
    return 0

df["deal_int"] = df["deal"].apply(deal_to_int)

deal_rate = (
    df.groupby(["mode", "transparency"])["deal_int"]
      .mean()
      .reset_index(name="deal_rate")
)

plt.figure(figsize=(8, 4))
sns.barplot(
    data=deal_rate,
    x="transparency",
    y="deal_rate",
    hue="mode"
)
plt.ylim(0, 1)
plt.title("Deal Rate by Transparency and Bargaining Mode")
plt.ylabel("Probability of Deal")
plt.xlabel("Transparency Regime")
plt.show()